# CS453 - Two-Week Individual Project Workbook

**Name:** [إنتصار محمد معيتيق]  
**Student ID:** [09225030]  
**Project:** [Cleaning Robot]

Read the two-week student guide first. This workbook supplies only map data and a variant helper, not search or agent solutions. Complete your own explanations and implementations. Save this notebook in your private GitHub repository as project.ipynb and follow docs/github-colab-guide.md. Submit the repository and final commit links in Classroom. The full course project continues after the professor returns.


## 1. Task, PEAS and environment


**Task.** After working hours, a cleaning robot starts at location S of a campus map and must clean one assigned target room. The map is a graph: nodes are locations (hallways, rooms) and edges are open roads. One road is closed in each student's variant.

**Input:** the road graph, start location S, the target room, and the assigned variant (from the student ID).  
**Output:** a plan, i.e. a sequence of `move` actions ending with one `clean` action, plus its verified execution in the simulator.  
**Success criteria:** the target room is cleaned (task status = done) using the minimum number of moves; the plan contains only legal actions.

**PEAS**

| | |
|---|---|
| Performance measure | Target room cleaned; fewest moves (each move costs 1); no illegal actions |
| Environment | Graph of 8 campus locations with one closed road; empty after hours |
| Actuators | Move to an adjacent location; clean the current room |
| Sensors | Current location, adjacent locations, whether the target is already cleaned |

**Environment properties (for this simulation)**

- **Fully observable:** the robot knows the whole map, its location and the task status.
- **Deterministic:** a legal move always reaches the chosen neighbour; cleaning always succeeds.
- **Sequential:** the moves taken now decide where the robot can clean later; cleaning is only possible after the right moves.
- **Static:** nothing changes while the robot plans (no people, no new closures).
- **Discrete:** a finite set of locations and discrete actions.
- **Single agent:** one robot, nobody else acts.

**Three scope limits:** (1) one target room only, no multiple dirty rooms or priorities; (2) no battery, time, or dirt-level model, and every road has the same cost 1; (3) no physical robot, sensors noise, or moving obstacles, and the map never changes during a run.

## 2. Representation

- **State:** `(location, cleaned)`, where `cleaned` is the task status of the target room (`False` / `True`). Location alone is not enough, because being at the target is different from having cleaned it.
- **Initial state:** `("S", False)`. The task is **not** complete at the start: `cleaned` is `False`, and the target is never S in any variant.
- **Legal actions in `(loc, cleaned)`:**
  - `("move", n)` for every neighbour `n` of `loc` in the graph (closed roads are not in the graph, so they are illegal).
  - `("clean", loc)` only if `loc == target` and `cleaned` is `False`.
- **Service legality and repeated service:** cleaning elsewhere is illegal; cleaning the target a second time is also illegal (it is already clean), so the simulator rejects it with a clear error.
- **Result / transition:** `move n` gives `(n, cleaned)`; `clean` gives `(loc, True)`. An illegal action raises an error and the state is unchanged.
- **Goal test:** `cleaned == True` (this implies the robot did go to the target, because cleaning is only legal there).
- **Costs:** every action costs 1. The single `clean` action is in every solution, so comparing plans is the same as comparing the number of moves.

In [16]:
# Supplied data/helper: cite CS453 two-week project workbook if adapted.
from copy import deepcopy
import platform

REFERENCE_GRAPH = {
    "S": ["A", "B"],
    "A": ["S", "C", "D"],
    "B": ["S", "E"],
    "C": ["A", "F"],
    "D": ["A", "G"],
    "E": ["B", "G"],
    "F": ["C", "G"],
    "G": ["D", "E", "F"],
}
VARIANTS = [("A", "D", "G"), ("B", "E", "F"),
            ("C", "F", "E"), ("E", "G", "D")]

def make_variant(student_id):
    if not str(student_id).strip():
        raise ValueError("Enter your student ID first")
    variant = sum(ord(ch) for ch in str(student_id).strip()) % 4
    graph = deepcopy(REFERENCE_GRAPH)
    u, v, goal = VARIANTS[variant]
    graph[u].remove(v)
    graph[v].remove(u)
    return {"variant": variant, "graph": graph, "start": "S",
            "goal": goal, "removed_edge": (u, v)}

STUDENT_ID = "09225030"  # TODO: your student ID as text, preserving leading zeros.
if STUDENT_ID.strip():
    assigned = make_variant(STUDENT_ID)
    print(assigned)
else:
    print("Unfinished template: enter your ID to generate your variant.")
print("Python:", platform.python_version())

{'variant': 1, 'graph': {'S': ['A', 'B'], 'A': ['S', 'C', 'D'], 'B': ['S'], 'C': ['A', 'F'], 'D': ['A', 'G'], 'E': ['G'], 'F': ['C', 'G'], 'G': ['D', 'E', 'F']}, 'start': 'S', 'goal': 'F', 'removed_edge': ('B', 'E')}
Python: 3.13.18


### Supplied heuristic data for Task 6

These are shortest distances in the reference graph with all original roads open. Your assigned variant removes a road, so those relaxed-world distances cannot overestimate the remaining route cost. They are supplied task data; students implement the actual BFS and A* algorithms. For goal F: S=3, A=2, B=3, C=1, D=2, E=2, F=0, G=1. No extra hand-trace task is required.

In [17]:
# Supplied heuristic data: h is distance in the original graph before a road closure.
# Removing roads cannot make a route shorter, so these values remain lower bounds.
# h(goal) = 0. These tables apply to the original graph and its edge-deletion variants.
# If you add shortcuts/change costs, re-justify h or use h = 0 instead.
HEURISTICS = {'A': {'A': 0, 'B': 2, 'C': 1, 'D': 1, 'E': 3, 'F': 2, 'G': 2, 'S': 1}, 'B': {'A': 2, 'B': 0, 'C': 3, 'D': 3, 'E': 1, 'F': 3, 'G': 2, 'S': 1}, 'C': {'A': 1, 'B': 3, 'C': 0, 'D': 2, 'E': 3, 'F': 1, 'G': 2, 'S': 2}, 'D': {'A': 1, 'B': 3, 'C': 2, 'D': 0, 'E': 2, 'F': 2, 'G': 1, 'S': 2}, 'E': {'A': 3, 'B': 1, 'C': 3, 'D': 2, 'E': 0, 'F': 2, 'G': 1, 'S': 2}, 'F': {'A': 2, 'B': 3, 'C': 1, 'D': 2, 'E': 2, 'F': 0, 'G': 1, 'S': 3}, 'G': {'A': 2, 'B': 2, 'C': 2, 'D': 1, 'E': 1, 'F': 1, 'G': 0, 'S': 3}, 'S': {'A': 1, 'B': 1, 'C': 2, 'D': 2, 'E': 2, 'F': 3, 'G': 3, 'S': 0}}
print("Heuristic tables available for targets:", ", ".join(sorted(HEURISTICS)))


Heuristic tables available for targets: A, B, C, D, E, F, G, S


## 4. Implement and test the simulator

The simulator works on states `(location, cleaned)`. `legal_actions` lists what the robot may do, `result` applies one action and rejects illegal ones with a `ValueError`, `goal_test` checks the task status, and `execute_mission` takes a route found by search, appends the `clean` action, and runs everything step by step. Functions never modify the graph passed in, so tests do not affect each other. Search plans the route; the simulator verifies it.


In [18]:
def check_location(graph, loc):
    if loc not in graph:
        raise ValueError(f"Invalid location: {loc!r}")

def legal_actions(state, graph, target):
    loc, cleaned = state
    check_location(graph, loc)
    actions = [("move", n) for n in sorted(graph[loc])]
    if loc == target and not cleaned:
        actions.append(("clean", loc))
    return actions

def result(state, action, graph, target):
    # Returns the new state, or raises ValueError for an illegal action.
    if action not in legal_actions(state, graph, target):
        raise ValueError(f"Illegal action {action} in state {state}")
    loc, cleaned = state
    kind, arg = action
    if kind == "move":
        return (arg, cleaned)
    return (loc, True)          # kind == "clean"

def goal_test(state):
    return state[1]             # task status: target room cleaned

def step_cost(state, action):
    return 1

def mission_actions(path, target):
    # Moves along the route, then the single service action.
    return [("move", n) for n in path[1:]] + [("clean", target)]

def execute_mission(path, graph, start, target):
    state = (start, False)
    trace, total = [], 0
    for action in mission_actions(path, target):
        new_state = result(state, action, graph, target)
        total += step_cost(state, action)
        trace.append((state, action, new_state))
        state = new_state
    return state, trace, total

# ---- simulator tests (independent of the student ID) ----
def expect_error(label, fn):
    try:
        fn()
    except ValueError as e:
        print(f"OK   {label}: rejected ({e})")
    else:
        raise AssertionError(f"{label}: should have been rejected")

G = deepcopy(REFERENCE_GRAPH)
before = deepcopy(G)
T = "G"

assert legal_actions(("S", False), G, T) == [("move", "A"), ("move", "B")]
assert ("clean", "G") in legal_actions(("G", False), G, T)
assert ("clean", "G") not in legal_actions(("G", True), G, T)
assert not goal_test(("S", False)), "task must not be complete at the start"

expect_error("clean in the wrong room", lambda: result(("S", False), ("clean", "S"), G, T))
expect_error("move to a non-neighbour", lambda: result(("S", False), ("move", "G"), G, T))
expect_error("invalid location", lambda: legal_actions(("Z", False), G, T))
expect_error("clean twice", lambda: result(("G", True), ("clean", "G"), G, T))

final, trace, total = execute_mission(["S", "A", "D", "G"], G, "S", T)
assert final == ("G", True) and goal_test(final) and total == 4
print("OK   full mission S-A-D-G + clean ->", final, "cost", total)
assert G == before, "graph was mutated"
print("OK   graph not mutated")

OK   clean in the wrong room: rejected (Illegal action ('clean', 'S') in state ('S', False))
OK   move to a non-neighbour: rejected (Illegal action ('move', 'G') in state ('S', False))
OK   invalid location: rejected (Invalid location: 'Z')
OK   clean twice: rejected (Illegal action ('clean', 'G') in state ('G', True))
OK   full mission S-A-D-G + clean -> ('G', True) cost 4
OK   graph not mutated


## 5. Uninformed search: BFS

**Why uninformed.** BFS uses no knowledge about where the goal is; it only uses the graph structure. It expands the start, then all nodes 1 move away, then 2 moves away, and so on.

**How it works here.** The frontier is a FIFO queue (`deque`). The `parent` dictionary does two jobs: it is the set of already-reached states (so cycles in the map are never re-explored) and it stores each node's parent so the route is rebuilt backwards from the goal. Neighbours are visited in sorted order so results are reproducible.

**Why it is optimal here.** Every move costs 1, and the FIFO queue reaches nodes in non-decreasing number of moves. The first time the goal is reached, it is reached with the fewest moves possible.

**Edge cases.** `start == goal` returns `[start]` (zero moves, then the robot cleans); an unreachable goal returns `None`; unknown locations raise `ValueError`. The route found is then executed in the simulator, including the `clean` action.

In [19]:
from collections import deque

def reconstruct(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def bfs(graph, start, goal):
    check_location(graph, start)
    check_location(graph, goal)
    if start == goal:
        return [start]
    frontier = deque([start])              # FIFO queue
    parent = {start: None}                 # reached states + parents
    while frontier:
        node = frontier.popleft()
        for nb in sorted(graph[node]):
            if nb not in parent:           # repeated-state check
                parent[nb] = node
                if nb == goal:             # unit costs: safe to stop here
                    return reconstruct(parent, nb)
                frontier.append(nb)
    return None                            # no route

# ---- edge cases (independent of the student ID) ----
assert bfs(REFERENCE_GRAPH, "S", "S") == ["S"]
assert bfs({"S": ["A"], "A": ["S"], "Z": []}, "S", "Z") is None
for bad in [("S", "Q"), ("Q", "S")]:
    try:
        bfs(REFERENCE_GRAPH, *bad)
    except ValueError as e:
        print("OK   invalid location rejected:", e)
print("OK   start=goal, no route and invalid location handled")

# ---- assigned mission ----
if STUDENT_ID.strip():
    m = assigned
    path = bfs(m["graph"], m["start"], m["goal"])
    print(f"Variant {m['variant']} | closed road {m['removed_edge']} | clean room {m['goal']}")
    print("BFS route:", " -> ".join(path), f"({len(path) - 1} moves)")
    final, trace, total = execute_mission(path, m["graph"], m["start"], m["goal"])
    for s, a, s2 in trace:
        print(f"  {s} --{a[0]} {a[1]}--> {s2}")
    print("Task complete:", goal_test(final), "| total cost:", total)
else:
    print("Enter your STUDENT_ID in the variant cell above to run your assigned mission.")


OK   invalid location rejected: Invalid location: 'Q'
OK   invalid location rejected: Invalid location: 'Q'
OK   start=goal, no route and invalid location handled
Variant 1 | closed road ('B', 'E') | clean room F
BFS route: S -> A -> C -> F (3 moves)
  ('S', False) --move A--> ('A', False)
  ('A', False) --move C--> ('C', False)
  ('C', False) --move F--> ('F', False)
  ('F', False) --clean F--> ('F', True)
Task complete: True | total cost: 4


## 6. Informed search: A*

**g and h.** `g(n)` is the cost already paid: the number of moves from S to `n` along the best known route. `h(n)` is the supplied estimate of the remaining moves from `n` to the assigned room (`HEURISTICS[goal][n]`). A* always expands the frontier node with the smallest `f(n) = g(n) + h(n)`, using a priority queue (`heapq`). `best_g` records the cheapest cost found to each node; a better route replaces the parent. The goal is tested **when it is removed from the frontier**, not when it is generated, because only then is its `g` guaranteed to be optimal.

**Why h is admissible on the provided variant.** `h(n)` is the true shortest distance in the original graph with all roads open. A variant only removes one road, and removing a road can never make a route shorter, so the real remaining cost in the variant is at least `h(n)`. So h never overestimates. It is also consistent (`h(n) <= 1 + h(n')` for every open road `n -> n'`), because it is an exact distance in a graph that contains all the variant's roads. The code below checks both properties on your assigned graph.

**When h would need revising.** If the map gets shortcuts or new roads, if some roads cost more or less than 1, if the cleaning goal changes (a different table is needed for each target), or if cleaning gets its own cost or several rooms must be cleaned, the supplied table may overestimate. Then it must be recomputed and re-justified, or replaced with `h = 0` (which makes A* behave like uniform-cost search and stays admissible).

**Compared with BFS.** All costs are 1, so A* and BFS return routes with the same number of moves; the routes themselves can differ when several shortest routes exist. A* usually expands fewer nodes because h guides it toward the goal.

In [20]:
import heapq

def astar(graph, start, goal, h):
    check_location(graph, start)
    check_location(graph, goal)
    best_g = {start: 0}
    parent = {start: None}
    frontier = [(h[start], 0, start)]      # (f, g, node) priority queue
    expanded = 0
    while frontier:
        f, g, node = heapq.heappop(frontier)
        if g > best_g[node]:               # stale queue entry
            continue
        if node == goal:                   # goal test on removal
            return reconstruct(parent, node), g, expanded
        expanded += 1
        for nb in sorted(graph[node]):
            new_g = g + 1
            if nb not in best_g or new_g < best_g[nb]:
                best_g[nb] = new_g
                parent[nb] = node
                heapq.heappush(frontier, (new_g + h[nb], new_g, nb))
    return None, None, expanded            # no route

# ---- edge cases (independent of the student ID) ----
h_ref = HEURISTICS["G"]
assert astar(REFERENCE_GRAPH, "G", "G", h_ref)[0] == ["G"]
assert astar({"S": ["A"], "A": ["S"], "Z": []}, "S", "Z", {"S": 0, "A": 0, "Z": 0})[0] is None
try:
    astar(REFERENCE_GRAPH, "Q", "G", h_ref)
except ValueError as e:
    print("OK   invalid location rejected:", e)
print("OK   start=goal, no route and invalid location handled")

# ---- assigned mission ----
if STUDENT_ID.strip():
    m = assigned
    h = HEURISTICS[m["goal"]]
    path, cost, expanded = astar(m["graph"], m["start"], m["goal"], h)
    print(f"Variant {m['variant']} | closed road {m['removed_edge']} | clean room {m['goal']}")
    print("A* route:", " -> ".join(path), f"({cost} moves, {expanded} nodes expanded)")

    # verify admissibility and consistency on the assigned graph
    for n in m["graph"]:
        true_cost = len(bfs(m["graph"], n, m["goal"])) - 1
        assert h[n] <= true_cost, f"h not admissible at {n}"
        for nb in m["graph"][n]:
            assert h[n] <= 1 + h[nb], f"h not consistent on road {n}-{nb}"
    print("Checked: h is admissible and consistent on the assigned graph.")

    bfs_path = bfs(m["graph"], m["start"], m["goal"])
    assert len(bfs_path) - 1 == cost, "BFS and A* disagree on the move count"
    print("BFS and A* agree on the minimum move count:", cost)

    final, trace, total = execute_mission(path, m["graph"], m["start"], m["goal"])
    for s, a, s2 in trace:
        print(f"  {s} --{a[0]} {a[1]}--> {s2}")
    print("Task complete:", goal_test(final), "| total cost:", total)
else:
    print("Enter your STUDENT_ID in the variant cell above to run your assigned mission.")

OK   invalid location rejected: Invalid location: 'Q'
OK   start=goal, no route and invalid location handled
Variant 1 | closed road ('B', 'E') | clean room F
A* route: S -> A -> C -> F (3 moves, 3 nodes expanded)
Checked: h is admissible and consistent on the assigned graph.
BFS and A* agree on the minimum move count: 3
  ('S', False) --move A--> ('A', False)
  ('A', False) --move C--> ('C', False)
  ('C', False) --move F--> ('F', False)
  ('F', False) --clean F--> ('F', True)
Task complete: True | total cost: 4


## 10. Sources, assistance and questions

**Adapted course material.** The reference graph, `VARIANTS`, `make_variant` and the `HEURISTICS` tables are supplied by the CS453 two-week project workbook and are used unchanged. BFS (FIFO queue plus parent map) and A* (priority queue on f = g + h with best-cost tracking) are my implementations of the standard algorithms. The simulator for the cleaning-robot state `(location, cleaned)` was written for this project. [edit: if you used the course graph-search / informed-search study notebooks, name them here and say what you changed]

**AI assistance.** [edit to match what you actually did] I used Claude (Anthropic) to help draft the PEAS/representation text and the simulator, BFS and A* code. I verified the work by reading and re-explaining every function, running the edge-case tests in sections 4-6 (start = goal, no route, invalid location, illegal and repeated cleaning), checking that BFS and A* return the same number of moves, and checking that h is admissible and consistent on my assigned graph.

**Unresolved questions.** [complete, or write "none"]

I can explain BFS, A*, the heuristic and my simulator after the professor returns.

## GitHub + Colab workflow

1. On the public starter, use Use this template > Create a new repository, choose Private, then invite the professor using the announced GitHub username.
2. Your copy already includes project.ipynb. Open it in Colab using authorized GitHub access or download/upload.
3. Save each work session explicitly to GitHub: File > Save a copy in/to GitHub, correct repo/branch and project.ipynb. Drive autosave alone does not update GitHub.
4. If integration fails, download the notebook from Colab, upload it to GitHub to replace project.ipynb, and commit. No terminal or embedded tokens are needed.
5. Commit real progress throughout the project. Complete README.md using the supplied template.
6. Reopen the committed notebook in a fresh Colab session and verify it runs. Submit repo URL + final commit URL in Classroom and Turn in.

See the student guide pages 5-6 and docs/github-colab-guide.md for details. The professor must accept the invitation before viewing private work; do not wait for acceptance to begin.


## 11. Submission checklist

- [ ] Identity, chosen project and generated variant recorded
- [ ] Task 1: PEAS and environment explained
- [ ] Task 2: state, legal actions, transition, goal and costs defined
- [ ] Task 4: legal movement, service and task goal implemented
- [ ] Task 5: working BFS and assigned mission output
- [ ] Task 6: working A*, admissible heuristic explanation and assigned mission output
- [ ] Sources and assistance disclosed
- [ ] README with task/variant/files/dependencies/run steps
- [ ] Clean restart/run-all checked and saved
- [ ] project.ipynb and README.md saved in my private GitHub repository
- [ ] Professor invited; repository URL and final commit URL submitted in Classroom

Original task numbers are retained for reference. Tasks 3, 7, 8 and 9 are removed. There is no Day 7 checkpoint.